# Глубинное обучение для текстовых данных, ФКН ВШЭ
## Домашнее задание 3: Уменьшение размеров модели
### Оценивание и штрафы

Максимально допустимая оценка за работу — __16 баллов__.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом для пользоваться запрещается даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке. Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

__Мягкий дедлайн 22.11.26 23:59__ \
__Жесткий дедлайн 24.11.26 23:59__

### О задании

В этом задании вам предстоит научиться решать задачу Named Entity Recognition (NER) на самом популярном датасете – [CoNLL-2003](https://paperswithcode.com/dataset/conll-2003). В вашем распоряжении будет предобученный BERT, который вам необходимо уменьшить с минимальными потерями в качестве до размера 20М параметров. Для этого вы самостоятельно реализуете факторизацию эмбеддингов, дистилляцию, шеринг параметров и так далее.

В этом задании вам придется проводить довольно много экспериментов, поэтому мы рекомендуем не писать весь код в тетрадке, а завести разные файлы для отдельных логических блоков и скомпоновать все в виде проекта. Это позволит вашему ноутбуку не разрастаться и сильно облегчит задачу и вам, и проверяющим. Так же постарайтесь логгировать все ваши эксперименты на wandb или других платформах, чтобы ничего не потерялось.

В этом задании __запрещается использовать агентов для написания кода__, но разрешается для помощи в чтении статей и осознании методов.

### Оценивание
Оценка за это домашнее задание будет формироваться из оценки за __задания__ и за __отчет__, в котором от вас требуется написать о проделанной работе. За отчет можно получить до 2-х баллов, однако в случае отсутствия отчета баллы за соответствующие задания не будут ставиться. Задания делятся на две части: _номерные_ и _на выбор_. За _номерные_ можно получить в сумме 6 баллов, за задания _на выбор_ можно получить до 8. То есть за все дз можно получить 16 баллов (но не радуйтесь рано, это не так просто). Все, что вы наберете свыше 10, будет считаться бонусами.

### О датасете

Named Entity Recognition – это задача классификации токенов по классам сущностей. В CoNLL-2003 для именования сущностей используется маркировка **BIO** (Beggining, Inside, Outside), в которой метки означают следующее:

- *B-{метка}* – начало сущности *{метка}*
- *I-{метка}* – продолжнение сущности *{метка}*
- *O* – не сущность

Существуют так же и другие способы маркировки, например, BILUO. Почитать о них можно [тут](https://en.wikipedia.org/wiki/Inside–outside–beginning_(tagging)) и [тут](https://www.youtube.com/watch?v=dQw4w9WgXcQ).

Всего в датасете есть 9 разных меток.
- O – слову не соответствует ни одна сущность.
- B-PER/I-PER – слово или набор слов соответстует определенному _человеку_.
- B-ORG/I-ORG – слово или набор слов соответстует определенной _организации_.
- B-LOC/I-LOC – слово или набор слов соответстует определенной _локации_.
- B-MISC/I-MISC – слово или набор слов соответстует сущности, которая не относится ни к одной из предыдущих. Например, национальность, произведение искусства, мероприятие и т.д.

Приступим!

Начнем с загрузки и предобработки датасета.

In [1]:
from datasets import load_dataset

dataset = load_dataset("ashaba1in/conll2003")

dataset = dataset.remove_columns(["id", "pos_tags", "chunk_tags"])
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3453
    })
})

In [2]:
from datasets import load_dataset

dataset2 = load_dataset("hgissbkh/conll2003-en")
dataset2

DatasetDict({
    train: Dataset({
        features: ['words', 'ner'],
        num_rows: 14042
    })
    validation: Dataset({
        features: ['words', 'ner'],
        num_rows: 3252
    })
    test: Dataset({
        features: ['words', 'ner'],
        num_rows: 3454
    })
})

In [3]:
dataset['train'][1]

{'tokens': ['Peter', 'Blackburn'], 'ner_tags': [1, 2]}

In [4]:
dataset['train'][0]

{'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [5]:
label_names = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [6]:
words = dataset["train"][0]["tokens"]
labels = dataset["train"][0]["ner_tags"]

for i in range(len(words)):
    print(f'{words[i]}\t{label_names[labels[i]]}')

EU	B-ORG
rejects	O
German	B-MISC
call	O
to	O
boycott	O
British	B-MISC
lamb	O
.	O


### Предобработка

На протяжении всего домашнего задания мы будем использовать _cased_ версию BERT, то есть токенизатор будет учитывать регистр слов. Для задачи NER регистр важен, так как имена и названия организаций или предметов искусства часто пишутся с большой буквы, и будет глупо прятать от модели такую информацию.

In [7]:
from transformers import set_seed

set_seed(42)

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

При токенизации слова могут разделиться на несколько токенов (как слово `Fischler` из примера ниже), из-за чего появится несоответствие между числом токенов и меток. Это несоответствие нам придется устранить вручную.

In [9]:
example = dataset["train"][12]
words = example["tokens"]
tags = [label_names[t] for t in example["ner_tags"]]
tokenized_text = tokenizer(example["tokens"], is_split_into_words=True)

print('Слова: ', words)
print('Токены:', tokenized_text.tokens())
print('Метки:', tags)

Слова:  ['Only', 'France', 'and', 'Britain', 'backed', 'Fischler', "'s", 'proposal', '.']
Токены: ['[CLS]', 'Only', 'France', 'and', 'Britain', 'backed', 'Fi', '##sch', '##ler', "'", 's', 'proposal', '.', '[SEP]']
Метки: ['O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'O', 'O', 'O']


__Задание 1 (1 балл).__ Токенизируйте весь датасет и для каждого текста выравните токены с метками так, чтобы каждому токену соответствовала одна метка. При этом важно сохранить нотацию BIO. И не забудьте про специальные токены! Должно получиться что-то такое:

In [10]:
def align_labels_with_tokens(labels, tokenized_text):
    tokenized_labels = []
    prev_id = None

    for word_id in tokenized_text.word_ids():
        if word_id is None:
            label = -100
        else:
            label = labels[word_id]
            if word_id == prev_id and label in [1, 3, 5, 7]:
                label += 1

        tokenized_labels.append(label)
        prev_id = word_id

    return tokenized_labels

In [11]:
aligned_labels = align_labels_with_tokens(example["ner_tags"], tokenized_text)
tags = [label_names[t] if t > -1 else t for t in aligned_labels]
print("Выровненные метки:", aligned_labels)
print("Выровненные названия меток:", tags)

Выровненные метки: [-100, 0, 5, 0, 5, 0, 1, 2, 2, 0, 0, 0, 0, -100]
Выровненные названия меток: [-100, 'O', 'B-LOC', 'O', 'B-LOC', 'O', 'B-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', -100]


In [12]:
# your code here
def tokenize_one(line):
    tokenized = tokenizer(line["tokens"], is_split_into_words=True, truncation=True)
    tokenized["labels"] = align_labels_with_tokens(line["ner_tags"], tokenized)
    return tokenized


tokenized_dataset = dataset.map(tokenize_one, remove_columns=dataset["train"].column_names)

### Метрика

Для оценки качества NER обычно используют F1 меру с микро-усреднением. Мы загрузим ее из библиотеки `seqeval`. Функция `f1_score` принимает два 2d списка с правильными и предсказанными метками, записаными текстом, и возвращает для них значение F1. Вы можете использовать ее с параметрами по умолчанию.

In [13]:
# ! pip install seqeval

In [14]:
from seqeval.metrics import f1_score

Особенность подсчета F1 для NER заключается в том, что в некоторых ситуациях неправильные ответы могут засчитываться как правильные. Например, если модель предсказала `['I-PER', 'I-PER']`, то мы можем догадаться, что на самом деле должно быть `['B-PER', 'I-PER']`, так как сущность не может начинаться с `I-`. Функция `f1_score` учитывает это и поэтому работает только с текстовыми представлениями меток.

### Модель

В качестве базовой модели мы возьмем `bert-base-cased`. Как вы понимаете, он не обучался на задачу NER. Поэтому прежде чем приступать к уменьшению размера BERT, его необходимо дообучить.

__Задание 2 (1 балл)__ Дообучите `bert-base-cased` на нашем датасете с помощью обычного fine-tuning. У вас должно получиться хотя бы 0.9 F1 на тестовой выборке. Заметьте, что чем выше качество большой модели, тем лучше будет работать дистиллированный ученик. Для обучения можно использовать `Trainer` из Hugging Face.

In [15]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained('bert-base-cased', num_labels=len(label_names))

print('Число параметров:', sum(p.numel() for p in model.parameters()))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Число параметров: 107726601


In [16]:
# your code here
from transformers import DataCollatorForTokenClassification, Trainer, TrainingArguments, set_seed




def get_f1(res):
    preds = res.predictions.argmax(axis=-1)
    target = []
    predicted = []

    for guess, labels in zip(preds, res.label_ids):
        target.append([label_names[i] for i in labels[labels != -100]])
        predicted.append([label_names[i] for i in guess[labels != -100]])

    return {"f1": f1_score(target, predicted)}


teacher_args = TrainingArguments(
    output_dir="checkpoints/teacher",
    num_train_epochs=5,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",
    seed=42
)

set_seed(teacher_args.seed)
trainer = Trainer(
    model=model,
    args=teacher_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=get_f1
)


In [17]:
trainer.train()
trainer.save_model("best_weights/teacher")
test = trainer.evaluate(tokenized_dataset["test"])
print(f"тестовый F1 учителя: {test['eval_f1']:.4f}")
del trainer, model


Epoch,Training Loss,Validation Loss,F1
1,0.078600,0.064219,0.916130
2,0.039100,0.070504,0.936231
3,0.024600,0.065100,0.938772
4,0.013700,0.069697,0.944013
5,0.006100,0.074105,0.947105


тестовый F1 учителя: 0.9040


### Факторизация матрицы эмбеддингов

Можно заметить, что на данный момент матрица эмбеддингов занимает $V \cdot H = 28996 \cdot 768 = 22.268.928$ параметров. Это aж пятая часть от всей модели! Давайте попробуем что-то с этим сделать. В модели [ALBERT](https://arxiv.org/pdf/1909.11942.pdf) предлагается факторизовать матрицу эмбеддингов в произведение двух небольших матриц. Таким образом, параметры эмбеддингов будут содержать $V \cdot E + E \cdot H$ элементов, что гораздо меньше $V \cdot H$, если $H \gg E$. Авторы выбирают $E = 128$, однако ничего не мешает нам взять любое другое значение. Например, выбрав $H = 64$, мы уменьшим число параметров примерно на 20М.

__Задание 3 (1 балл).__ Напишите класс-обертку над слоем эмбеддингов, который реализует факторизацию на две матрицы, и дообучите факторизованную модель. Заметьте, обе матрицы можно инициализировать с помощью SVD разложения (а лучше [FWSVD](https://arxiv.org/pdf/2207.00112)!), чтобы начальное приближение было хорошим. Это сэкономит очень много времени на дообучении. С рангом разложения $H = 64$ у вас должно получиться F1 больше 0.87.

In [18]:
# your code here
import torch
from torch import nn
from dataclasses import replace


class FactorizedEmbedding(nn.Module):
    def __init__(self, old_embedding, rank=64):
        super().__init__()

        self.embedding = nn.Embedding(
            old_embedding.num_embeddings,
            rank,
            padding_idx=old_embedding.padding_idx
        )
        self.projection = nn.Linear(rank, old_embedding.embedding_dim, bias=False)

        with torch.no_grad():
            u, s, v = torch.svd_lowrank(old_embedding.weight.float(), q=rank + 16, niter=4)
            self.embedding.weight.copy_(u[:, :rank] * s[:rank])
            self.projection.weight.copy_(v[:, :rank])

        self.to(device=old_embedding.weight.device, dtype=old_embedding.weight.dtype)

    def forward(self, input_ids):
        return self.projection(self.embedding(input_ids))


In [19]:
set_seed(teacher_args.seed)
fact_embs_model = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
old_embedding = fact_embs_model.get_input_embeddings()
fact_embs_model.set_input_embeddings(FactorizedEmbedding(old_embedding, rank=64))
fact_embs_trainer = Trainer(
    model=fact_embs_model,
    args=replace(teacher_args, output_dir="checkpoints/factorized"),
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=get_f1
)


In [20]:
fact_embs_trainer.train()
fact_embs_trainer.save_model("best_weights/factorized")
test = fact_embs_trainer.evaluate(tokenized_dataset["test"])
print(f"тестовый f1 с факторизацией: {test['eval_f1']:.4f}")
del fact_embs_trainer, fact_embs_model, old_embedding


Epoch,Training Loss,Validation Loss,F1
1,0.080100,0.092611,0.895303
2,0.044900,0.103230,0.916013
3,0.025500,0.099750,0.926303
4,0.013900,0.105631,0.929794
5,0.006400,0.109663,0.931273


тестовый f1 с факторизацией: 0.8854


### Дистилляция знаний

Дистилляция знаний – это парадигма обучения, в которой знания модели-учителя дистиллируются в модель-ученика. Учеником может быть произвольная модель меньшего размера, решающая ту же задачу, однако обычно ученик имеет ту же архитектуру, что и учитель. При дистилляции используются два функционала ошибки:

1. Стандартная кросс-энтропия.
1. Функция, задающая расстояние между распределениями предсказаний учителя и ученика. Чаще всего используют KL-дивергенцию.

Для того, чтобы распределение предсказаний учителя не было вырожденным, к softmax добавляют температуру больше 1, например, 2 или 5.   
__Важно:__ при делении логитов на температуру значения градиентов уменьшаются в $\tau^2$ раз (проверьте это!). Поэтому для возвращения их в изначальный масштаб ошибку надо домножить на $\tau^2$. Подробнее об этом можно почитать в разделе 2.1 [оригинальной статьи](https://arxiv.org/pdf/1503.02531).

<img src="https://intellabs.github.io/distiller/imgs/knowledge_distillation.png" width="800">

__Задание 4 (3 балла).__ Реализуйте метод дистилляции знаний, изображенный на картинке. Для подсчета ошибки между предсказаниями ученика и учителя используйте KL-дивергенцию [`nn.KLDivLoss(reduction="batchmean")`](https://pytorch.org/docs/stable/generated/torch.nn.KLDivLoss.html) (обратите внимание на вормат ее входов). Для получения итоговой ошибки суммируйте мягкую ошибку с жесткой.   
В качестве учителя используйте дообученный BERT из задания 2. В качестве ученика возьмите необученную модель с размером __не больше 20M__ параметров. Вы можете использовать факторизацию матрицы эмбеддингов для уменьшения числа параметров. Если вы все сделали правильно, то на тестовой выборке вы должны получить значение F1 не меньше 0.7. Вам должно хватить примерно 20к итераций обучения для этого. Если у вас что-то не получается, то можно ориентироваться на статью про [DistilBERT](https://arxiv.org/abs/1910.01108) и на [эту статью](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT).

__Важно:__
* Не забывайте добавлять _warmup_ при обучении ученика.
* Не забывайте переводить учителя в режим _eval_.

In [21]:
# your code here
from transformers import BertConfig, BertForTokenClassification

student_args = replace(
    teacher_args,
    output_dir="checkpoints/student",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    max_steps=25000,
    warmup_steps=1000
)

config = BertConfig(
    vocab_size=len(tokenizer),
    num_labels=len(label_names),
    hidden_size=256,
    num_hidden_layers=6,
    num_attention_heads=4,
    intermediate_size=1024
)


In [22]:
class DistillationTrainer(Trainer):
    def __init__(self, teacher, temperature=2.0, **kwargs):
        super().__init__(**kwargs)
        self.teacher = teacher.to(self.args.device)
        self.teacher.eval()
        self.temperature = temperature
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        hard_loss = outputs.loss
        teacher_inputs = {k: v for k, v in inputs.items() if k != "labels"}

        with torch.no_grad():
            teacher_logits = self.teacher(**teacher_inputs).logits

        mask = inputs["labels"] != -100
        t = self.temperature
        student_logits = outputs.logits[mask] / t
        teacher_logits = teacher_logits[mask] / t
        soft_loss = self.kl(student_logits.log_softmax(dim=-1), teacher_logits.softmax(dim=-1))
        loss = hard_loss + t**2 * soft_loss

        if return_outputs:
            return (loss, outputs)
        else:
            return loss


In [23]:
set_seed(student_args.seed)
teacher = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
student = BertForTokenClassification(config)
print(f"Параметров у ученика: {sum(p.numel() for p in student.parameters()):,}")
distill_trainer = DistillationTrainer(
    teacher=teacher,
    model=student,
    args=student_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=get_f1
)


max_steps is given, it will override any value given in num_train_epochs


Параметров у ученика: 12,295,945


In [24]:
distill_trainer.train()
distill_trainer.save_model("best_weights/student")
test = distill_trainer.evaluate(tokenized_dataset["test"])
print(f"тестовый f1 ученика: {test['eval_f1']:.4f}")
del distill_trainer, student, teacher


Epoch,Training Loss,Validation Loss,F1
1,4.640800,1.872292,0.409173
2,1.376800,1.298101,0.536482
3,0.719100,1.042946,0.609443
4,0.458200,0.998012,0.658926
5,0.331300,0.913099,0.709549
6,0.234100,0.965102,0.703820
7,0.185300,0.922546,0.730592
8,0.153800,0.981656,0.717555
9,0.126000,0.927008,0.748260
10,0.106100,1.104651,0.742433


тестовый f1 ученика: 0.7189


# Задания на выбор

Как вы понимаете, есть еще довольно много разных способов уменьшить обученную модель. В этой секции вам предлагается реализовать разные техники на выбор. За каждую из них можно получить разное количество балов в зависимости от сложности. Успешность реализации будет оцениваться как по коду, так и по качеству на тестовой выборке. Все баллы за это дз, которые вы наберете сверх 10, будут считаться бонусными.   
В задании 4 вы обучали модель с ограничением числа параметров в 20М. При реализации техник из этой секции придерживайтесь такого же ограничения. Это позволит честно сравнивать методы между собой и делать правильные выводы. Напишите в отчете обо всем, что вы попробовали.

* __Шеринг весов (1 балл).__ В модификации BERT [ALBERT](https://arxiv.org/pdf/1909.11942) помимо факторизации эмбеддингов предлагается шерить веса между слоями. То есть разные слои используют одни и те же веса. Такая техника эвивалентна применению одного и того же слоя несколько раз. Она позволяет в несколько раз уменьшить число параметров и не сильно потерять в качестве.
* __Шеринг весов++ (0.5 балл).__ Вместо того, чтобы просто применять одни и те же слои, можно дообучить LoRA для каждого слоя, чтобы слои могли специализироваться. Ровно так делают в [этой статье](https://arxiv.org/pdf/2410.20672) от DeepMind.
* __Приближение промежуточных слоев (1.5 балла).__ Мы обсуждали, что помимо приближения выходов модели ученика к выходам модели учителя, можно приближать выходы промежуточных слоев. В [этой работе](https://www.researchgate.net/publication/375758425_Knowledge_Distillation_Scheme_for_Named_Entity_Recognition_Model_Based_on_BERT) подробно написано, как это можно сделать. Еще __+1 балл__ за реализацию метода из [MiniLMv2](https://arxiv.org/pdf/2012.15828).
* __Прунинг (3 балла).__ В методе [SparseGPT](https://arxiv.org/abs/2301.00774) предлагается подход, удаляющий веса модели один раз после обучения. При этом оказывается возможным удалить до половины всех весов без потери в качестве. Математика, стоящаяя за техникой, довольно сложная, однако общий подход простой – будем удалять веса в каждом слое по отдельности, при удалении части весов слоя, остальные веса будут перенастраиваться так, чтобы общий выход слоя не изменился. В качестве бейзлайна реализуйте более простой метод [Wanda](https://arxiv.org/pdf/2306.11695).
* __Инициализация ученика прунингом (2 балла).__ Обучение ученика с нуля занимает достаточно много времени. Вместо этого, ученика можно инициализоровать с помощью прунинга учителя, а затем дообучить с помощью дистилляции. Впервые так сделали в [Minitron](https://arxiv.org/pdf/2408.11796), а после в Ministral 3, Llama 3.2, Falcon3 и т.д. Важная проблема этого подхода заключается в том, что слои могут пруниться неравномерно, из-за чего при использовании модели будет сложнее распределять нагрузку. Но если задача – просто уменьшить размер, то это отличный способ.



In [25]:
import comet_ml
import pandas as pd
from transformers import TrainerCallback
from student_models import share_layers, add_lora

comet_ml.login(project_name="hw3-distillation-25k-seed42")


In [26]:
class CometLogger(TrainerCallback):
    def __init__(self, exp):
        self.exp = exp

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs:
            self.exp.log_metric("train/loss", logs["loss"], step=state.global_step)
        if "eval_f1" in logs:
            self.exp.log_metric("eval/f1", logs["eval_f1"], step=state.global_step)


In [27]:
#1
set_seed(student_args.seed)
teacher = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
shared_student = share_layers(BertForTokenClassification(config))
num_params = sum((p.numel() for p in shared_student.parameters()))
sharing_args = replace(student_args, output_dir="checkpoints/sharing", save_safetensors=False)
exp = comet_ml.start(project_name="hw3-distillation-25k-seed42", mode="create")
exp.set_name("student-sharing")
try:
    exp.log_parameters({"num_parameters": num_params, "num_unique_layers": 3})
    sharing_trainer = DistillationTrainer(
        teacher=teacher,
        model=shared_student,
        args=sharing_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        processing_class=tokenizer,
        compute_metrics=get_f1,
        callbacks=[CometLogger(exp)]
    )
    sharing_trainer.train()
    shared_student.config.save_pretrained("best_weights/sharing")
    torch.save(shared_student.state_dict(), "best_weights/sharing/model.pt")
    test = sharing_trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
    
    exp.log_metric("test/f1", test["test_f1"])

    print(f"Параметров: {num_params:,}")
    print(f"Тестовый F1: {test['test_f1']:.4f}")
finally:
    exp.end()
del sharing_trainer, shared_student, teacher


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/hw3-distillation-25k-seed42/570ebba4e1e44454b126c751d7743ea9

COMET INFO: Couldn't find a Git repository in '/home/ktkram/hse_26_27/NLP_hse_hws/hw3' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
max_steps is given, it will override any value given in num_train_epochs


Epoch,Training Loss,Validation Loss,F1
1,4.640300,1.918495,0.390658
2,1.398600,1.315488,0.546107
3,0.711800,0.992738,0.607803
4,0.464100,0.998707,0.679272
5,0.329700,0.920810,0.706584
6,0.230500,0.917469,0.734951
7,0.184100,1.011243,0.718812
8,0.147200,0.964559,0.750505
9,0.117500,0.995274,0.744250
10,0.096500,0.956305,0.754951


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : student-sharing
COMET INFO:     url                   : https://www.comet.com/scfeel/hw3-distillation-25k-seed42/570ebba4e1e44454b126c751d7743ea9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     eval/f1 [29]    : (0.3906584582441114, 0.7915737298636927)
COMET INFO:     test/f1         : 0.7274308575404418
COMET INFO:     train/loss [50] : (0.0103, 4.6403)
COMET INFO:   Others:
COMET INFO:     Name : student-sharing
COMET INFO:   Parameters:
COMET INFO:     num_parameters    : 9926665
COMET INFO:     num_unique_layers : 3
COMET INFO:   Uploads:
COMET INFO:     environment details : 1
COMET INFO:     filename            : 1
COMET INFO:     ins

Параметров: 9,926,665
Тестовый F1: 0.7274


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.


In [28]:
#2
teacher = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
lora_config = BertConfig.from_pretrained("best_weights/sharing")
weights = torch.load("best_weights/sharing/model.pt", map_location="cpu", weights_only=True)
results = []
best_f1 = -1

for lr in [1e-5, 5e-5, 1e-4, 2e-4]:
    set_seed(student_args.seed)
    model = share_layers(BertForTokenClassification(lora_config))
    model.load_state_dict(weights)
    model = add_lora(model, rank=32)
    lora_args = replace(
        student_args,
        output_dir=f"checkpoints/sharing_lora/{lr}",
        learning_rate=lr,
        max_steps=-1,
        num_train_epochs=3,
        warmup_steps=100,
        save_safetensors=False
    )
    lora_trainer = DistillationTrainer(
        teacher=teacher,
        model=model,
        args=lora_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        processing_class=tokenizer,
        compute_metrics=get_f1
    )
    if len(results) == 0:
        base_f1 = lora_trainer.evaluate()["eval_f1"]
        print(f"валидация f1 до обучения LoRA: {base_f1:.4f}")
    lora_trainer.train()
    f1 = lora_trainer.state.best_metric
    results.append({"lr": lr, "val_f1": f1, "delta": f1 - base_f1})
    if f1 > best_f1:
        best_f1 = f1
        best_lr = lr
        model.config.save_pretrained("best_weights/sharing_lora")
        torch.save(model.state_dict(), "best_weights/sharing_lora/model.pt")
    del lora_trainer, model

results = pd.DataFrame(results)
results.to_csv("best_weights/sharing_lora/results.csv", index=False)
print(results.to_string(index=False))
print(f"лучший LR: {best_lr}")
del teacher, weights


валидация f1 до обучения LoRA: 0.7916


Epoch,Training Loss,Validation Loss,Model Preparation Time,F1
1,0.011700,0.902545,0.001300,0.791773
2,0.010400,0.901434,0.001300,0.791608
3,0.009900,0.901107,0.001300,0.791674


Epoch,Training Loss,Validation Loss,F1
1,0.011800,0.899242,0.791870
2,0.010400,0.896997,0.792263
3,0.009800,0.896301,0.791512


Epoch,Training Loss,Validation Loss,F1
1,0.011800,0.897248,0.792032
2,0.010400,0.896289,0.791839
3,0.009800,0.895308,0.792241


Epoch,Training Loss,Validation Loss,F1
1,0.011800,0.895289,0.791680
2,0.010400,0.897012,0.790456
3,0.009700,0.895447,0.790276


     lr   val_f1    delta
0.00001 0.791773 0.000200
0.00005 0.792263 0.000689
0.00010 0.792241 0.000667
0.00020 0.791680 0.000107
лучший LR: 5e-05


In [29]:
results = pd.read_csv("best_weights/sharing_lora/results.csv")
best = results.loc[results["val_f1"].idxmax()]
lora_config = BertConfig.from_pretrained("best_weights/sharing_lora")
model = add_lora(share_layers(BertForTokenClassification(lora_config)), rank=32)
model.load_state_dict(torch.load(
    "best_weights/sharing_lora/model.pt",
    map_location="cpu",
    weights_only=True
))
args = replace(student_args, output_dir="checkpoints/sharing_lora_eval", save_safetensors=False)
trainer = Trainer(
    model=model,
    args=args,
    eval_dataset=tokenized_dataset["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,
    compute_metrics=get_f1
)
exp = comet_ml.start(project_name="hw3-distillation-25k-seed42", mode="create")
exp.set_name("student-sharing-lora")
try:
    num_params = sum(p.numel() for p in model.parameters())
    num_train = sum((p.numel() for p in model.parameters() if p.requires_grad))
    exp.log_parameters({
        "lr": best["lr"],
        "rank": 32,
        "epochs": 3,
        "warmup_steps": 100,
        "num_parameters": num_params,
        "trainable_parameters": num_train
    })
    train = trainer.evaluate(tokenized_dataset["train"], metric_key_prefix="train")
    test = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
    exp.log_metric("train/f1", train["train_f1"])
    exp.log_metric("eval/f1", best["val_f1"])
    exp.log_metric("test/f1", test["test_f1"])
    print(f"Параметров: {num_params:,}, обучаемых: {num_train:,}")
    print(f"Train F1: {train['train_f1']:.4f}")
    print(f"Validation F1: {best['val_f1']:.4f}")
    print(f"Тестовый F1: {test['test_f1']:.4f}")
finally:
    exp.end()
del trainer, model


max_steps is given, it will override any value given in num_train_epochs
COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/hw3-distillation-25k-seed42/397eeb8f7aa94011857d394f2871121c

COMET INFO: Couldn't find a Git repository in '/home/ktkram/hse_26_27/NLP_hse_hws/hw3' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : student-sharing-lora
COMET INFO:     url                   : https://www.comet.com/scfeel/hw3-distillation-25k-seed42/397eeb8f7aa94011857d394f2871121c
COMET INFO:   Metrics:
COMET INFO:     eval/f1  : 0.792262544432504
COMET INFO:     test/f1  : 0.7264979563440298
COMET INFO:     train/f1 : 0.9977450646698434
COMET INFO:   Others:
COMET INFO:     Name : student-sharing-lora
COMET INFO:   Parameters:
COMET INFO:     epochs               : 3
COMET INFO:     lr                   : 5e-05
COMET INFO:     num_parameters       : 10123273
COMET INFO:     rank                 : 32
COMET INFO:     trainable_parameters : 196608
COMET INFO:     warmup_steps     

Параметров: 10,123,273, обучаемых: 196,608
Train F1: 0.9977
Validation F1: 0.7923
Тестовый F1: 0.7265


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET INFO: Please wait for metadata to finish uploading (timeout is 3600 seconds)
COMET INFO: Uploading 1 metrics, params and output messages


In [30]:
#3
from distillation import HiddenTrainer

set_seed(student_args.seed)
teacher = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
student = BertForTokenClassification(config)
student.projection = nn.Linear(config.hidden_size, teacher.config.hidden_size, bias=False)

hidden_args = replace(student_args, output_dir="checkpoints/hidden")
num_params = sum(p.numel() for p in student.parameters())
print(f"параметров с проекцией: {num_params:,}")


параметров с проекцией: 12,492,553


In [31]:
exp = comet_ml.start(project_name="hw3-distillation-25k-seed42", mode="create")
exp.set_name("student-hidden")

try:
    exp.log_parameters({"num_parameters": num_params, "alpha": 1.0, "temperature": 2.0})
    hidden_trainer = HiddenTrainer(
        teacher=teacher,
        model=student,
        args=hidden_args,
        alpha=1.0,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=DataCollatorForTokenClassification(tokenizer),
        processing_class=tokenizer,
        compute_metrics=get_f1,
        callbacks=[CometLogger(exp)]
    )
    hidden_trainer.train()

    train = hidden_trainer.evaluate(tokenized_dataset["train"], metric_key_prefix="train")
    test = hidden_trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
    exp.log_metric("train/f1", train["train_f1"])
    exp.log_metric("best/f1", hidden_trainer.state.best_metric)
    exp.log_metric("test/f1", test["test_f1"])

    del student.projection
    hidden_trainer.save_model("best_weights/hidden")

    print(f"Параметров итоговой модели: {sum(p.numel() for p in student.parameters()):,}")
    print(f"Train F1: {train['train_f1']:.4f}")
    print(f"Validation F1: {hidden_trainer.state.best_metric:.4f}")
    print(f"Тестовый F1: {test['test_f1']:.4f}")
finally:
    exp.end()

del hidden_trainer, student, teacher


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/hw3-distillation-25k-seed42/6da29bffd20c4ef98b82c7624ac33ffa

COMET INFO: Couldn't find a Git repository in '/home/ktkram/hse_26_27/NLP_hse_hws/hw3' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
max_steps is given, it will override any value given in num_train_epochs


Epoch,Training Loss,Validation Loss,F1
1,5.464600,2.575563,0.334745
2,1.933500,1.712396,0.540986
3,1.182200,1.399767,0.595369
4,0.869600,1.397260,0.659046
5,0.731100,1.354737,0.688626
6,0.599800,1.266559,0.709363
7,0.538500,1.282095,0.724619
8,0.498000,1.373968,0.724212
9,0.458200,1.227653,0.728551
10,0.434100,1.356177,0.759255


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : student-hidden
COMET INFO:     url                   : https://www.comet.com/scfeel/hw3-distillation-25k-seed42/6da29bffd20c4ef98b82c7624ac33ffa
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     best/f1         : 0.8106955380577427
COMET INFO:     eval/f1 [29]    : (0.3347454584619803, 0.8106955380577427)
COMET INFO:     test/f1         : 0.7400241837968561
COMET INFO:     train/f1        : 0.9976179335573611
COMET INFO:     train/loss [50] : (0.265, 5.4646)
COMET INFO:   Others:
COMET INFO:     Name : student-hidden
COMET INFO:   Parameters:
COMET INFO:     alpha          : 1.0
COMET INFO:     num_parameters : 12492553
COMET INFO:     temper

Параметров итоговой модели: 12,295,945
Train F1: 0.9976
Validation F1: 0.8107
Тестовый F1: 0.7400


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET INFO: Please wait for assets to finish uploading (timeout is 10800 seconds)
COMET INFO: Still uploading 2 file(s), remaining 37.10 KB/37.10 KB


In [32]:
#4
from copy import deepcopy
from pruned_init import prune_student

set_seed(student_args.seed)
teacher = AutoModelForTokenClassification.from_pretrained("best_weights/teacher")
teacher.to(student_args.device).eval()
pruned_args = replace(student_args, output_dir="checkpoints/pruned")
collator = DataCollatorForTokenClassification(tokenizer)

model = deepcopy(teacher)
layers = list(model.bert.encoder.layer)
trainer = Trainer(
    model=model,
    args=pruned_args,
    eval_dataset=tokenized_dataset["validation"],
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=get_f1
)
best_f1 = -1

for start in range(7):
    keep = list(range(start)) + list(range(start + 6, 12))
    model.bert.encoder.layer = nn.ModuleList([layers[i] for i in keep])
    model.config.num_hidden_layers = 6
    f1 = trainer.evaluate()["eval_f1"]
    print(f"Удаляем блоки {start + 1}-{start + 6}: validation F1 = {f1:.4f}")
    if f1 > best_f1:
        best_f1 = f1
        blocks = keep

print("Оставляем блоки:", [i + 1 for i in blocks])
del trainer, model, layers


max_steps is given, it will override any value given in num_train_epochs


Удаляем блоки 1-6: validation F1 = 0.0369
Удаляем блоки 2-7: validation F1 = 0.2040
Удаляем блоки 3-8: validation F1 = 0.1911
Удаляем блоки 4-9: validation F1 = 0.1859
Удаляем блоки 5-10: validation F1 = 0.2158
Удаляем блоки 6-11: validation F1 = 0.1806
Удаляем блоки 7-12: validation F1 = 0.0813
Оставляем блоки: [1, 2, 3, 4, 11, 12]


In [33]:
sample = tokenized_dataset["train"].shuffle(seed=student_args.seed).select(range(256))

In [34]:
student = prune_student(
    teacher,
    BertConfig.from_pretrained("best_weights/student"),
    blocks,
    sample,
    collator
)
num_params = sum(p.numel() for p in student.parameters())
print(f"Параметров: {num_params:,}")


Параметров: 12,295,945


In [35]:
exp = comet_ml.start(project_name="hw3-distillation-25k-seed42", mode="create")
exp.set_name("student-pruned")

try:
    exp.log_parameters({"num_parameters": num_params, "blocks": blocks, "calibration_size": 256})
    pruned_trainer = DistillationTrainer(
        teacher=teacher,
        model=student,
        args=pruned_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=collator,
        processing_class=tokenizer,
        compute_metrics=get_f1,
        callbacks=[CometLogger(exp)]
    )
    before = pruned_trainer.evaluate()["eval_f1"]
    print(f"валидация f1 до обучения: {before:.4f}")
    exp.log_metric("before/f1", before)

    pruned_trainer.train()
    pruned_trainer.save_model("best_weights/pruned")

    test = pruned_trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
    exp.log_metric("best/f1", pruned_trainer.state.best_metric)
    exp.log_metric("test/f1", test["test_f1"])
    print(f"Лучший validation F1: {pruned_trainer.state.best_metric:.4f}")
    print(f"Тестовый F1: {test['test_f1']:.4f}")
finally:
    exp.end()

del pruned_trainer, student, teacher


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/hw3-distillation-25k-seed42/f54c46c5a4474be29cd8173402233e92

COMET INFO: Couldn't find a Git repository in '/home/ktkram/hse_26_27/NLP_hse_hws/hw3' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
max_steps is given, it will override any value given in num_train_epochs


валидация f1 до обучения: 0.0125


Epoch,Training Loss,Validation Loss,Model Preparation Time,F1
1,3.711100,0.964829,0.001300,0.655954
2,0.751000,0.626599,0.001300,0.784137
3,0.389100,0.577149,0.001300,0.805273
4,0.251600,0.623186,0.001300,0.814893
5,0.182800,0.621713,0.001300,0.830105
6,0.131200,0.640596,0.001300,0.811724
7,0.112200,0.596354,0.001300,0.833887
8,0.097500,0.626845,0.001300,0.850969
9,0.072400,0.615817,0.001300,0.851938
10,0.078500,0.665909,0.001300,0.852820


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : student-pruned
COMET INFO:     url                   : https://www.comet.com/scfeel/hw3-distillation-25k-seed42/f54c46c5a4474be29cd8173402233e92
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     before/f1       : 0.012506180379838873
COMET INFO:     best/f1         : 0.8772075974675108
COMET INFO:     eval/f1 [30]    : (0.012506180379838873, 0.8772075974675108)
COMET INFO:     test/f1         : 0.8187755812940868
COMET INFO:     train/loss [50] : (0.0079, 3.7111)
COMET INFO:   Others:
COMET INFO:     Name : student-pruned
COMET INFO:   Parameters:
COMET INFO:     blocks           : [0, 1, 2, 3, 10, 11]
COMET INFO:     calibration_size : 256
C

Лучший validation F1: 0.8772
Тестовый F1: 0.8188


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
